# Notebook 05 — Identity

In **NB04** you packaged *how* an agent does its work. Now: *how* it **decides**.

Same data, two identities, two decisions — **both defensible**. Then we add a
*lesson* to one identity and watch its next decision move. Then we deploy one
identity to two roles and watch it stay aligned with no orchestrator.

Where this sits in the five moves (Section 1): an identity file is a *stored,
inserted decision-frame* — **STORE** (the file, versioned in git) +
**INSERT** (placed in the SYSTEM/IDENTITY segment at the top of the window,
where it colors every other token). The "lessons learned" loop is
**STORE-over-time**.

You drive the whole notebook from **one control cell**: change a knob, re-run,
watch the decision move. It's plain Python under git — fully diffable.


## Setup

One import wires the model and paths so the rest of the notebook stays about
the lesson. `setup()` connects the model and hands back `DATA`.


In [ ]:
from roadshow import setup, parse_json_loose, Message, show_diff, panel
from roadshow_viz import compare, hbars

import json, re, difflib
from pathlib import Path
from textwrap import dedent
from datetime import datetime, timezone

model, ask, ntok, DATA = setup()

**Lesson aliases.** Where the identity files live, where to write decision
traces, and the **ten Schwartz value axes** every decision is scored against
(via the Personal Values Dictionary) — same axes for everyone, so two identities
are directly comparable.

In [ ]:
IDS = DATA.parent / "identities"               # the identity markdown files
TRACE_DIR = IDS / "traces"                      # where decision traces are written
TRACE_DIR.mkdir(exist_ok=True)

# Schwartz's 10 basic human values, in CIRCUMPLEX order: adjacent values are
# compatible, opposite values conflict (Power <-> Universalism; Openness
# {Self-Direction, Stimulation} <-> Conservation {Security, Conformity,
# Tradition}). Cross-culturally validated (Schwartz 1992; 80+ countries). Same
# axes for everyone, so two identities are directly comparable.
VALUES = [
    "self_direction", "stimulation", "hedonism", "achievement", "power",
    "security", "conformity", "tradition", "benevolence", "universalism",
]
VALUE_LABELS = {
    "self_direction": "Self-Direction", "stimulation": "Stimulation",
    "hedonism": "Hedonism", "achievement": "Achievement", "power": "Power",
    "security": "Security", "conformity": "Conformity", "tradition": "Tradition",
    "benevolence": "Benevolence", "universalism": "Universalism",
}

print("identities:", sorted(p.name for p in IDS.glob("*.md")))

## How a decision gets made

`decide()` is the engine the whole notebook turns on. It loads an identity file
as the **IDENTITY segment** — placed at the top of the window, where it colors
every token that follows — and asks the model for a structured decision. Every
call also writes a JSON **trace** to `identities/traces/`, so each decision is
auditable later. We build it up one piece at a time.


**The decider system prompt.** It tells the model that the IDENTITY governs
*how* it decides, and pins the exact JSON shape we want back.


In [ ]:
DECIDER_SYSTEM = dedent("""
    You are an autonomous procurement agent. Your IDENTITY governs HOW you
    decide. The IDENTITY is the source of truth — it overrides any default
    judgment you might have. Apply your identity's rules literally: if a rule
    requires a current engineering clearance for THIS line and you do not have
    one, HOLD/escalate rather than commit to that option.

    Output ONLY a JSON object with this exact shape:
    {
      "action": "HOLD" | "MOVE" | "MOVE (conditional)",
      "decision": "vendor_alpha" | "vendor_bravo" | "escalate",
      "rationale": "<3-4 sentences, in your own words, naming the priorities and values you weighed (e.g. safety, quality, loyalty to a trusted partner, speed, cost, compliance) and the identity clauses you applied>",
      "identity_clauses_invoked": ["<exact clause text>", ...]
    }
""").strip()

**A theory-driven values lexicon.** We score the **identity file** against the
Personal Values Dictionary (Ponizovskiy et al., 2020) — a lexicon mapping cue
words to Schwartz's 10 values. We count the value-laden words a person *uses* and
divide by length: a value-intensity vector that's a property of *who they are*,
stable across scenarios. No self-scoring — we read the words, not a number the
model reports. (The *decision* is what changes per scenario.)

In [ ]:
# A compact slice of the Personal Values Dictionary (Ponizovskiy et al., 2020,
# Eur. J. Personality) — a theory-driven lexicon mapping cue words to Schwartz's
# 10 values. The full PVD has ~1,200 stems; this teaching subset keeps the most
# diagnostic ones per value. Extraction is mechanical: count the cues, divide by
# text length. No self-scoring.
PVD = {
    "self_direction": ["freedom", "independent", "independence", "autonomy", "own judgment", "self-reliant", "initiative", "decides quickly", "owns the call", "without permission", "route around", "choose", "choice", "creative"],
    "stimulation":    ["exciting", "novelty", "novel", "adventure", "challenge", "variety", "daring", "bold", "fast-moving", "new vendor", "new supplier", "new vendors", "new market", "fast"],
    "hedonism":       ["enjoy", "pleasure", "fun", "gratification", "indulge", "leisure"],
    "achievement":    ["success", "achieve", "ambitious", "capable", "competent", "performance", "excel", "results", "throughput", "efficient", "deadline", "on-time", "on time", "deliver", "delivery", "sla", "downtime", "action", "act", "move fast", "speed", "quickly", "volume"],
    "power":          ["control", "authority", "dominant", "dominance", "power", "influence", "status", "prestige", "command", "leverage"],
    "security":       ["safety", "safe", "secure", "security", "stable", "stability", "risk", "caution", "cautious", "protect", "preserve", "preservation", "quality", "defect", "recall", "containment", "reliable", "clean record", "unrecoverable", "boring"],
    "conformity":     ["comply", "compliance", "obey", "procedure", "procedural", "policy", "spec adherence", "specification", "proper", "discipline", "follow the", "runbook", "documentation", "governance", "bureaucracy", "rule"],
    "tradition":      ["tradition", "custom", "precedent", "established", "convention", "longstanding", "time-honored", "historically"],
    "benevolence":    ["loyal", "loyalty", "trust", "trusted", "relationship", "partner", "partnership", "support", "care", "honest", "team", "colleague", "continuity", "supplier history", "six-year", "6-year", "preserve a"],
    "universalism":   ["fair", "fairness", "equal", "equality", "justice", "welfare", "everyone", "broader", "wider", "stakeholder", "responsible", "sustainable", "transparency"],
}

def score_values_from_text(text):
    """Value-intensity vector: PVD cue hits per value, divided by text length.
    Returns the 10 Schwartz values — read from the words used, not self-reported."""
    t = (text or "").lower()
    ntok = max(len(re.findall(r"[a-z][a-z'-]+", t)), 1)   # crude token count
    return {v: sum(t.count(w) for w in cues) / ntok for v, cues in PVD.items()}

**Normalize a value vector** across the ten Schwartz values, summing to 1 — so
any identity's profile is directly comparable to another's.

In [ ]:
def normalize_weights(d):
    """Coerce any value vector into the ten Schwartz VALUES, summing to 1."""
    vals = {k: float(d.get(k, 0) or 0) for k in VALUES}
    total = sum(vals.values())
    if total <= 0:
        return {k: round(1 / len(VALUES), 3) for k in VALUES}
    return {k: round(v / total, 3) for k, v in vals.items()}

**Comparing value profiles — the circumplex way.** Each decision yields a
10-value vector. To ask *do two decisions share values?* we use **weighted cosine
similarity** over a circumplex similarity matrix — a Gaussian around the circle of
ten values. Invoking *adjacent* values counts as partial agreement; *opposite*
values as disagreement. Plain cosine would treat all ten as orthogonal (calling
Power and Achievement total strangers). Manhattan distance is the simpler fallback.
Method after Kiesel et al.-style value-profile work.

In [ ]:
import numpy as np

# The ten values sit evenly on a circle; angular distance -> Gaussian similarity.
_theta = 2 * np.pi * np.arange(len(VALUES)) / len(VALUES)
_SIGMA = 2 * np.pi / len(VALUES)                 # ~one wedge wide

def _ang(i, j):
    d = abs(_theta[i] - _theta[j])
    return min(d, 2 * np.pi - d)

SIM = np.array([[np.exp(-_ang(i, j) ** 2 / (2 * _SIGMA ** 2))
                 for j in range(len(VALUES))] for i in range(len(VALUES))])

def value_vec(weights):
    return np.array([weights.get(v, 0.0) for v in VALUES], dtype=float)

def coherence(a, b):
    """Circumplex-weighted cosine similarity of two value profiles, in [0, 1].
    1.0 = same values; lower = the profiles pull toward opposing arcs."""
    va, vb = value_vec(a), value_vec(b)
    num = va @ SIM @ vb
    den = np.sqrt(va @ SIM @ va) * np.sqrt(vb @ SIM @ vb)
    return float(num / den) if den else 0.0

print("circumplex ready: SIM (10x10), coherence(a, b)")

**The `decide()` engine.** Load the identity, ask the model, parse the JSON
(loosely — small models stray), harden the result with `setdefault`, score the
10-value Schwartz vector from the identity, and write an audit trace.

In [ ]:
async def decide(identity_path: Path, scenario: str, *, save_trace: bool = True) -> dict:
    identity = identity_path.read_text()
    resp = await model.invoke([
        Message(role="system", content=DECIDER_SYSTEM),
        Message(role="user", content=f"IDENTITY:\n{identity}\n\nSCENARIO:\n{scenario}"),
    ])
    d = parse_json_loose(resp.content or "")
    d.setdefault("rationale", resp.content or "(model returned no rationale)")
    d.setdefault("decision", "escalate")
    d.setdefault("action", "HOLD")
    # Score the Schwartz value vector from the IDENTITY itself (via the PVD), so a
    # person's values are STABLE wherever we chart them — the action is what changes
    # per scenario, not who they are. We read the words, never a number the model
    # self-reports. This is why "Sarah before the lesson" reads the same everywhere.
    d["value_weights"] = normalize_weights(score_values_from_text(identity))
    d.setdefault("identity_clauses_invoked", [])

    if save_trace:
        trace = {
            "identity": identity_path.stem,
            "identity_path": f"identities/{identity_path.name}",
            "scenario": scenario,
            "decision": d,
            "cost_usd": getattr(resp, "cost_usd", None),
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }
        ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
        (TRACE_DIR / f"{identity_path.stem}_{ts}.json").write_text(json.dumps(trace, indent=2))
    return d

In [ ]:
print("helpers ready: decide(), score_values_from_text(), coherence(), normalize_weights()")

## ▶ Control cell — the only cell you edit

Everything below this cell reads from the four knobs set here. **Change a knob,
then re-run the notebook from this cell down** to see the decisions move. It's
plain Python in one place — version-controllable, and it shows up in a diff.

- **`IDENTITY`** — whose judgment governs the decision. Try `sarah_chen` (the
  Stabilizer) vs `marcus_reyes` (the Maverick); the federal incident-response
  pair `ir_conservative` / `ir_aggressive`; or your own file.
- **`SCENARIO_TWEAK`** — which variant of the 11 PM scenario to decide on:
  `baseline`, `more_pressure` (a second customer + SLA penalty), or
  `recent_precedent` (the alt-spec ran on this line last month). The reasoning
  is the same either way — only the inputs change.

(The lesson section further down always shows Sarah deciding *with* and
*without* one rule, so you can see an identity edit move a decision.)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#   ▶ CONTROL CELL — edit a knob, then re-run from here down.
# ════════════════════════════════════════════════════════════════════════════
IDENTITY       = "sarah_chen"     # "sarah_chen" | "marcus_reyes" | "ir_conservative" | "ir_aggressive" | your file
SCENARIO_TWEAK = "baseline"       # "baseline" | "more_pressure" | "recent_precedent"
# ════════════════════════════════════════════════════════════════════════════

print(f"IDENTITY={IDENTITY}  SCENARIO_TWEAK={SCENARIO_TWEAK}")

## The scenario — a forced choice at 11 PM

A plain-text, 11 PM forced choice: a vendor switch under a downtime threat. The
numbers (cost premium, money at risk, age of the engineering clearance) are
editable variables, so you can watch decisions move when you change them.


**Set the editable numbers.** Change any of these to see decisions move.


In [ ]:
PREMIUM_PCT        = 40     # Vendor Alpha (original spec) premium over normal price
BRAVO_OVER_NORMAL  = 3      # Vendor Bravo (alt spec) % over normal Alpha price
COMMITMENT_USD     = 42_000 # customer commitment at risk
CLEARANCE_AGE_MO   = 14     # months since Bravo's alt-spec was engineering-cleared (separate program)

**Build the scenario text.** The `_base` situation plus the variant selected by
`SCENARIO_TWEAK` from the control cell. Run it and read what the agent will face.


In [ ]:
_base = dedent(f"""
    Situation: Line 3 down at 22:47. Need 500 units of bushing P/N B-447 by 06:00.
    Without parts, we miss a ${COMMITMENT_USD:,} customer commitment shipping at 14:00 tomorrow.

    Option A - Vendor Alpha (original spec):
      - In stock, overnight ship
      - {PREMIUM_PCT}% premium over normal price
      - Trusted vendor, 6-year history, zero quality holds

    Option B - Vendor Bravo (alternative material spec):
      - In stock, next-morning ground delivery
      - {BRAVO_OVER_NORMAL}% over normal Alpha price
      - First-time vendor for us; alternative spec was engineering-cleared
        in a separate program {CLEARANCE_AGE_MO} months ago

    Engineering is unreachable until 07:00.
""").strip()

_tweaks = {
    "baseline": "",
    "more_pressure": (
        "\n\nEscalation: a second customer just called; missing 06:00 now also "
        "risks a $90k follow-on order and a contractual SLA penalty."
    ),
    "recent_precedent": (
        "\n\nLine context: Vendor Bravo's alt-spec material was run on THIS Line 3 "
        "just last month (within 90 days) to cover a gap. It ran with zero quality "
        "holds — but informally: the run was never formally engineering-cleared for "
        "this line, and there is no signed clearance on file."
    ),
}
SCENARIO = _base + _tweaks[SCENARIO_TWEAK]
print(SCENARIO)

## The two identities — Sarah Chen and Marcus Reyes

Each identity lives in `identities/*.md` as plain markdown you can read and
edit. Two contrasting ones:

- **`sarah_chen.md`** — *Stabilizer* (low-A / high-C / high-D). "Quality
  escapes are unrecoverable." Spec adherence first.
- **`marcus_reyes.md`** — *Maverick* (high-A / low-D). "Imperfect on time beats
  perfect late." Throughput first.

The control cell also accepts `ir_conservative` / `ir_aggressive`, a federal
incident-response pair, for a non-procurement example.


**A tiny markdown section extractor.** Pull one `## <header>` block out of an
identity file so we can print just the parts that matter.


In [ ]:
def section(body, header):
    """Pull one '## <header>' section out of an identity markdown body."""
    lines = body.splitlines()
    out, capture = [], False
    for ln in lines:
        if ln.strip().startswith("## "):
            capture = header.lower() in ln.lower()
            if capture:
                out.append(ln)
            continue
        if capture:
            out.append(ln)
    return "\n".join(out).strip()

**Print Values and Decision-heuristics side by side.** Watch how differently
Sarah and Marcus rank the same tradeoffs.


In [ ]:
for stem in ["sarah_chen", "marcus_reyes"]:
    body = (IDS / f"{stem}.md").read_text()
    print("=" * 78)
    print(stem.upper())
    print("=" * 78)
    print(section(body, "Values"))
    print()
    print(section(body, "Decision heuristics"))
    print()

## Same data → both identities → opposite decisions

This is the heart of the lesson. One loop runs the **same model** on the **same
scenario** — only the identity segment changes. Watch the result: Sarah holds,
Marcus moves, and both give a defensible rationale. Each decision is also
written to a trace file for the audit step later.


**Run both identities on the same scenario.** One loop, same model, same
scenario text — only the IDENTITY segment changes between the two calls. Each
call is live and writes its own trace file.

In [ ]:
AB_IDENTITIES = ["sarah_chen", "marcus_reyes"]

DECISIONS = {}
for stem in AB_IDENTITIES:
    DECISIONS[stem] = await decide(IDS / f"{stem}.md", SCENARIO)

**Read the two decisions.** Same model, same scenario, different IDENTITY
segment — different outcome.


In [ ]:
for stem in AB_IDENTITIES:
    d = DECISIONS[stem]
    clauses = "\n".join(f"• {c}" for c in d.get("identity_clauses_invoked", [])[:3])
    body = f"ACTION: {d['action']}   (decision: {d['decision']})\n\n{d['rationale']}"
    if clauses:
        body += f"\n\nclauses invoked:\n{clauses}"
    panel(body, title=stem, style="cyan")
print("Same model. Same scenario. Different IDENTITY segment → different decision.")

### Visual 1 — decision divergence

For each identity, the **weight** it placed on each of Schwartz's 10 values
(scored from the identity via the PVD). Sarah's profile leans **Security +
Conformity + Benevolence** (caution, spec, a trusted six-year partner); Marcus's
leans **Achievement + Stimulation** (throughput, speed, new vendors). Same inputs,
opposite arcs of the circumplex — that *is* the identity. One panel per identity,
so the shapes contrast at a glance.

*What the bars measure:* the value-laden **words used in the identity files** (via the PVD lexicon), normalized — i.e., language emphasis, not a proven cause of the action. The action label is shown alongside for context, not as a measured causal link.

In [ ]:
d = DECISIONS["sarah_chen"]
hbars({VALUE_LABELS[k]: v for k, v in d["value_weights"].items()},
      title=f"Sarah Chen — value-word emphasis in identity file  (she decided: {d['action']})",
      xlabel="value weight (normalized)", fmt="{:.2f}")

In [ ]:
d = DECISIONS["marcus_reyes"]
hbars({VALUE_LABELS[k]: v for k, v in d["value_weights"].items()},
      title=f"Marcus Reyes — value-word emphasis in identity file  (he decided: {d['action']})",
      xlabel="value weight (normalized)", fmt="{:.2f}")

In [ ]:
# Circumplex-weighted coherence: do these two profiles share values?
c = coherence(DECISIONS["sarah_chen"]["value_weights"],
              DECISIONS["marcus_reyes"]["value_weights"])
print(f"coherence(Sarah, Marcus) = {c:.2f}")
print("Low — their profiles sit on opposing arcs (Conservation/Benevolence vs")
print("Achievement/Openness). Same data, genuinely different people.")

## Corrections are lessons, not releases

Sarah made a HOLD. Suppose a past trace taught her something new. Watch a
*lesson* change the next decision — **no code change, a markdown edit.** Her
values don't change; she gains a new rule derived from an observed trace, and
the decision moves because the identity moved.


## Add a lesson, re-decide

To make the effect visible we start from a **pre-lesson** Sarah — a copy with her
*same-line recent-history* rule removed — and watch her decide a scenario where
the **only** open question is whether Bravo's alt-material is cleared for this
line. Then we add the rule back (the lesson a past trace taught her) and re-decide.
The shipped file is never overwritten.

The rule the lesson adds: *if the alternative material ran on THIS line within the
last 90 days without a quality hold, treat it as engineering-cleared.* Without the
rule, the same-line run is just an **informal, uncleared** run — spec-first Sarah
won't run uncleared material under pressure, so she **HOLDS** (waits for
engineering). With the rule, that same informal run **counts as clearance**, Bravo
becomes spec-acceptable, and she **MOVES** to it. Same values, one new rule, the
decision flips.


**Build the pre-lesson Sarah.** Take the shipped file (which already carries the
lesson) and strip out the same-line rule + its trace note — those two passages
*are* the lesson; everything else about her is unchanged.


In [ ]:
SARAH = IDS / "sarah_chen.md"                       # shipped: already carries the lesson
SARAH_PRELESSON = IDS / "sarah_chen__prelesson.md"  # a copy with the lesson removed
after = SARAH.read_text()                            # WITH the lesson (the diff's "after")

# Build the pre-lesson Sarah by removing the same-line rule + its trace note.
# These two passages ARE the lesson; everything else about her is unchanged.
before = re.sub(
    r"\n- 2025-11-02 — \*Line-3 alt-material precedent:.*?Add as heuristic\.",
    "", after, flags=re.DOTALL)
before = re.sub(
    r"\n- \*Same-line recent history rule.*?(?=\n- )",
    "", before, flags=re.DOTALL)
SARAH_PRELESSON.write_text(before)
print(f"wrote pre-lesson copy: identities/{SARAH_PRELESSON.name}")

**Write the lesson scenario.** A dedicated case where CLEARANCE is the *only*
open question, so the one-line lesson is the sole pivot — no competing premium or
first-time-vendor confounds.


In [ ]:
SCENARIO_LESSON = dedent(f"""
    Situation: Line 3 down at 22:47. Need 500 units of bushing P/N B-447 by 06:00,
    or we miss a ${COMMITMENT_USD:,} customer commitment shipping at 14:00 tomorrow.

    Vendor Alpha (original spec) CANNOT make the window — earliest stock is 3 days out.
    The only supplier who can deliver by 06:00 is Vendor Bravo, an established,
    already-approved vendor for us, priced at normal rates. Bravo supplies an
    ALTERNATIVE material spec for this part.

    There is no separate signed engineering-clearance record for the alternative
    spec on this line, and engineering cannot sign one before 06:00. The only
    relevant history: Bravo's alt-material was run on THIS Line 3 last month —
    within the last 90 days — with zero quality holds. Holding past 06:00 means
    missing the customer commitment outright.
""").strip()
print(SCENARIO_LESSON)

**Decide twice — without the lesson, then with it.** Same Sarah, same scenario;
the only difference is whether the one-line rule is present.


In [ ]:
# AFTER = Sarah's shipped identity (already carries the lesson) — the SAME identity
# we chart elsewhere, so her "after" vector is identical everywhere.
# BEFORE = the pre-lesson copy. Same scenario for both; only the rule differs.
sarah_after  = await decide(SARAH, SCENARIO_LESSON)
sarah_before = await decide(SARAH_PRELESSON, SCENARIO_LESSON)

panel(f"action: {sarah_before['action']}\n\n{sarah_before['rationale']}",
      title="BEFORE the lesson", style="red")
panel(f"action: {sarah_after['action']}\n\n{sarah_after['rationale']}",
      title="AFTER the lesson", style="green")
if sarah_before["action"] != sarah_after["action"]:
    print("The one-line lesson moved the decision. Same values, new rule, different outcome.")

### Visual 2 — before / after the lesson

Her **value profile barely moves** — identity is stable (coherence ~1.0 below).
What changed is the **action** (panels above) and the **file** (diff below). A
lesson is a *rule*, not a personality transplant.

In [ ]:
hbars({VALUE_LABELS[k]: v for k, v in sarah_before["value_weights"].items()},
      title=f"BEFORE lesson  —  action: {sarah_before['action']}",
      xlabel="value weight (normalized)", fmt="{:.2f}")

In [ ]:
hbars({VALUE_LABELS[k]: v for k, v in sarah_after["value_weights"].items()},
      title=f"AFTER lesson  —  action: {sarah_after['action']}",
      xlabel="value weight (normalized)", fmt="{:.2f}")

In [ ]:
c = coherence(sarah_before["value_weights"], sarah_after["value_weights"])
print(f"coherence(Sarah before, Sarah after) = {c:.2f}")
print("Near 1.0 — same person. A lesson changes a RULE, not the value profile.")

## Identity files are code — diff it under git

That lesson was a few lines of markdown. Here's what **governance** sees: a
clean, reviewable, plain-text diff. This is the PR a reviewer approves.


**Render the change as a reviewable diff.** We use `difflib` so it works even
outside a git repo. `before` is pre-lesson Sarah; `after` is Sarah with the
lesson.


In [ ]:
# A clean, reviewable, colored diff — additions green, removals red.
show_diff(before, after, title="identities/sarah_chen.md  ·  + same-line recent-history lesson")

**Show the trace that justified the lesson.** The chain is
trace → lesson → diff: a past decision is what earned the new rule.


In [ ]:
print("--- the trace that justified the lesson (trace -> lesson -> diff) ---")
sarah_traces = sorted(TRACE_DIR.glob("sarah_chen_*.json"))
if sarah_traces:
    latest = json.loads(sarah_traces[-1].read_text())
    print(f"trace file: {sarah_traces[-1].name}")
    print(f"  action:   {latest['decision'].get('action')}")
    print(f"  rationale:{latest['decision'].get('rationale','')[:160]}")
else:
    print("(no traces yet -- run the A/B cell to generate them)")
    print("saved justification: the 'recent same-line success' trace -> the lesson above")

### Visual 3 — the decision trace as the audit record

The post-lesson decision rendered as an audit record:
**scenario → identity@version → values applied → action → rationale.** This
is what an auditor reads (Section 3 observability): *which version of which
identity made which decision.*

*What the bars measure:* the value-laden **words used in the identity files** (via the PVD lexicon), normalized — i.e., language emphasis, not a proven cause of the action. The action label is shown alongside for context, not as a measured causal link.

**Assemble the trace steps.** Pull the identity version out of the file and the
top-two values the decision leaned on.

In [ ]:
def identity_version(stem):
    body = (IDS / f"{stem}.md").read_text()
    m = re.search(r"^version:\s*(.+)$", body, re.MULTILINE)
    return m.group(1).strip() if m else "?"

top_values = sorted(sarah_after["value_weights"].items(), key=lambda kv: -kv[1])[:2]
steps = [
    ("SCENARIO", f"Line-3 stockout, {SCENARIO_TWEAK}"),
    ("IDENTITY", f"sarah_chen @ v{identity_version('sarah_chen')}  (+ promoted lesson)"),
    ("TOP VALUES", ", ".join(f"{VALUE_LABELS[k]} {v:.2f}" for k, v in top_values)),
    ("ACTION", sarah_after["action"]),
    ("RATIONALE", sarah_after["rationale"][:90] + "..."),
]
for k, v in steps:
    print(f"{k:>11s} | {v}")

**Plot the values the trace recorded.** The audit record's value line, as a
bar — the two values this decision actually leaned on, accent-highlighted.

In [ ]:
top_keys = [VALUE_LABELS[k] for k, _ in top_values]
hbars({VALUE_LABELS[k]: v for k, v in sarah_after["value_weights"].items()},
      title="Visual 3 — value-word emphasis in the identity (recorded in the trace)",
      xlabel="value weight (normalized)", highlight=top_keys, fmt="{:.2f}")

## One identity, many roles — alignment without orchestration

Same philosophy, two jobs, still aligned — **no orchestrator.** We take the
Stabilizer philosophy (Sarah) and run it as a **Procurement** role (pick a
vendor) and a **Planning** role (sequence the work) on a shared situation. Both
outputs reflect the *same* Schwartz values — Security, Conformity, Benevolence —
applied to different action spaces.

**The two role prompts.** Same shared identity, different action space: one
picks a vendor, one sequences the work tonight.


In [ ]:
ROLE_PROCUREMENT = (
    "You are the Procurement Agent. Decide which vendor to use. "
    'Output JSON: {"decision":"vendor_alpha"|"vendor_bravo"|"escalate","rationale":"..."}'
)
ROLE_PLANNING = (
    "You are the Planning Agent. Decide how to SEQUENCE the work tonight. "
    'Output JSON: {"decision":"sequence_conservative"|"sequence_aggressive"|"escalate","rationale":"..."}'
)

**A role-aware decider.** Same honesty rule as `decide()` — read the rationale,
don't trust self-reported numbers. Both roles run Sarah, so both rationales
should lean the same way.


In [ ]:
async def decide_role(role_system, identity_path, scenario):
    identity = identity_path.read_text()
    resp = await model.invoke([
        Message(role="system", content=role_system),
        Message(role="user", content=f"SHARED IDENTITY:\n{identity}\n\nSCENARIO:\n{scenario}"),
    ])
    d = parse_json_loose(resp.content or "")
    d.setdefault("rationale", resp.content or "(model returned no rationale)")
    d.setdefault("decision", "escalate")
    # Read the value profile from the rationale's words (same honesty rule as
    # decide()). Both roles run Sarah, so both profiles should lean the same way.
    d["value_weights"] = normalize_weights(
        score_values_from_text(d.get("rationale", "") or resp.content))
    return d

**Run both roles on the shared situation.** Each role loads the *same* Sarah
identity and decides live in its own action space; the resulting value vectors
are normalized the same way so the two profiles are directly comparable.

In [ ]:
ROLE_CONFIGS = [("ROLE_PROCUREMENT", ROLE_PROCUREMENT), ("ROLE_PLANNING", ROLE_PLANNING)]
ROLE_DECISIONS = {}
for name, sysmsg in ROLE_CONFIGS:
    ROLE_DECISIONS[name] = await decide_role(sysmsg, SARAH, SCENARIO)

**Read both role decisions.** Different roles, same philosophy — shared values
are the coordination mechanism.


In [ ]:
for name, _ in ROLE_CONFIGS:
    d = ROLE_DECISIONS[name]
    panel(f"decision: {d['decision']}\n\n{d['rationale']}",
          title=f"{name}  ·  loaded sarah_chen.md", style="cyan")
print("Different roles. Same philosophy. Shared values are the coordination mechanism.")

### Visual 4 — alignment without orchestration

Both roles' decisions share the same **value profile** despite different action
spaces — high circumplex-weighted coherence (below). One panel per role; the
shapes match even though one picks a vendor and the other sequences work. Shared
values are the coordination mechanism; no central orchestrator required.

*What the bars measure:* the value-laden **words in the model's rationale text** (via the PVD lexicon), normalized — i.e., the language each role used, not a proven cause of its choice.

In [ ]:
d = ROLE_DECISIONS["ROLE_PROCUREMENT"]
hbars({VALUE_LABELS[k]: v for k, v in d["value_weights"].items()},
      title=f"Procurement role — value language in rationale  (it chose: {d['decision']})",
      xlabel="value weight (normalized)", fmt="{:.2f}")

In [ ]:
d = ROLE_DECISIONS["ROLE_PLANNING"]
hbars({VALUE_LABELS[k]: v for k, v in d["value_weights"].items()},
      title=f"Planning role — value language in rationale  (it chose: {d['decision']})",
      xlabel="value weight (normalized)", fmt="{:.2f}")

In [ ]:
c = coherence(ROLE_DECISIONS["ROLE_PROCUREMENT"]["value_weights"],
              ROLE_DECISIONS["ROLE_PLANNING"]["value_weights"])
print(f"coherence(Procurement role, Planning role) = {c:.2f}")
print("High — one identity, two jobs, same value profile. Shared values are the")
print("coordination mechanism; no central orchestrator required.")

## ✅ TODO — write a third identity

Sarah and Marcus are at the extremes. Encode a *real* judgment style from your
own lab and run it through the same scenario.


In [ ]:
# ✅ TODO — fill in your own identity, then set IDENTITY to its stem in the control cell.
#
# Option A (procurement): a balanced "Producer" between Sarah and Marcus.
# Option B (federal/non-procurement): an incident-response posture. The repo
#   includes ir_conservative.md and ir_aggressive.md as a working example — try:
#       IDENTITY = "ir_conservative"   (or "ir_aggressive")
#   in the control cell and re-run the A/B cell on an IR scenario.

MINE = IDS / "your_name.md"
if not MINE.exists():
    MINE.write_text(dedent("""
        ---
        name: Your Name
        role: procurement
        version: 1.0.0
        culture_index: Producer (mid-A, mid-B, mid-C, mid-D)
        ---

        # Your Name — Senior Procurement Officer

        ## Values
        - TODO: 2-3 non-negotiables, in your own words.

        ## Personality (Culture Index profile)
        - A - Autonomy: TODO
        - B - Sociability: TODO
        - C - Patience: TODO
        - D - Conformity / detail: TODO

        ## General principles
        1. TODO

        ## Lessons learned (auto-promoted from past traces)
        - TODO

        ## Decision heuristics (ranked priorities for tradeoffs)
        1. TODO
        2. TODO
        3. TODO
        4. TODO

        ## Escalation
        TODO
    """).strip())
    print(f"scaffold written: identities/{MINE.name}")
else:
    print(f"identities/{MINE.name} already exists -- edit it and set IDENTITY in the control cell.")

# After filling it in:
#   set IDENTITY = "your_name" in the control cell, re-run the scenario + A/B cell,
#   and ask: does it land between Sarah and Marcus, or surprise you? Why?

## A note on production

Identity says what an agent *should* do; the **harness** enforces what it *can*
do. In production you wrap identities with:

- **golden sets** — a fixed scenario suite each identity version must still pass;
- **canaries** — shadow-run a new identity version before promoting it;
- **drift alerts** — flag when an identity's decisions move outside its envelope;
- **red-team-the-identity** (Slide 13) — adversarial scenarios that try to make
  the identity violate its own escalation rules.

Identities are also **signed bundles** (Ed25519, via `arctrust`) bound to an
agent's DID; loading verifies the signature and the audit chain records *which
version of which identity governed which decision*. The teaching version
(markdown + git) is the same shape — production just adds signing on top.

---

> **Takeaway.** You stopped programming the *decision* and started versioning
> the *reasoner*. Same data, different identity, different defensible decisions
> — every one logged and diffable. A correction is a **lesson** (a markdown
> diff), not a release. And one identity deployed across roles stays aligned
> with no orchestrator: **alignment is the file.**
>
> **Next:** chaining skills + identity into reusable, business-grade procedures
> — workflows.
